In [ ]:
# === Cell 1/6: Clone hippovoice + GPU check ===
# Same pattern as colab.ipynb's Step 2 -- run after selecting
# Accelerator: GPU T4 x2 in Kaggle's notebook settings (top right).
# Private repo: add a GitHub PAT to Kaggle Secrets as GH_TOKEN, then enable it.
#
# Uses subprocess with check=True, not !magic -- !magic swallows a non-zero
# exit code silently (confirmed for real: a failed EasyEdit pip install was
# once logged as if it succeeded). check=True raises CalledProcessError
# instead -- the right behavior for THIS cell specifically, since nothing
# later can possibly work if the repo itself never cloned, so a loud
# immediate failure here (a normal red Jupyter error -- the log()/step()
# infra below doesn't exist yet at this point) is correct, not something to
# swallow and continue past.

import os, sys, subprocess

if os.path.exists('/kaggle/working'):
    REPO_DIR = '/kaggle/working/hippovoice'
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GH_TOKEN')
        CLONE_URL = f'https://{token}@github.com/shivansh193/hippovoice.git'
    except Exception:
        CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'
        print('No GH_TOKEN secret found -- will work for public repos only')
else:
    REPO_DIR = '/content/hippovoice'
    CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', CLONE_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    commit = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h'], text=True).strip()
except Exception:
    commit = 'unknown'
print(f'commit [{commit}]')

# Confirmed the hard way on this project's other Kaggle work: checking
# immediately after clone fails in seconds instead of hours into a run
# that turns out to be running on CPU. ROME/MEMIT edits involve real
# gradient descent (v_num_grad_steps=20 per edit) -- CPU would make even
# the single-edit sanity check in Cell 4 painfully slow, let alone a full
# benchmark.
import torch
assert torch.cuda.is_available(), (
    'No GPU detected -- set Accelerator to "GPU T4 x2" in Kaggle Settings '
    '(top-right of the notebook editor) before running anything below.'
)
print(f'GPU OK: {torch.cuda.get_device_name(0)}')
print('Ready.')


In [ ]:
# === Cell 2/6: Logging setup ===
# log(msg) prints AND appends to a log file, flushed + fsynced immediately
# (not buffered) so a crash doesn't lose the line that would have explained
# it. step(name) wraps a block of work: logs START/DONE automatically, and
# on any exception logs FAILED with the full traceback -- and, deliberately,
# does NOT re-raise. A "Save & Run All (Commit)" job stops at the first
# uncaught exception (confirmed the hard way on colab.ipynb -- a crashed
# cell took the whole commit down with it, including the Save Results cell
# that would have written the one real number from that run), so swallowing
# a failure here instead of re-raising is what lets Cell 6 still run and
# write a results file -- reporting the failure honestly rather than saying
# nothing at all.
#
# require(*names) exists because of a real, confirmed gap: a step whose
# prerequisite (editor, extraction_llm, ...) never got set because an
# EARLIER step already failed and got swallowed only surfaces several cells
# later as a bare `NameError`, with nothing pointing back at the actual
# root cause. Call this as the first line inside a step() block for
# anything it depends on.
#
# run_shell(cmd) exists because `!pip install ...` / `!git clone ...`
# shell-magic lines do NOT raise a Python exception on a non-zero exit
# code -- IPython just prints the error and moves on. A failed EasyEdit
# `pip install -r requirements.txt` was silently logged as DONE on a real
# run because of exactly this. Every shell command from Cell 3 onward runs
# through run_shell instead of !magic specifically so a real failure
# becomes a real exception step() can actually catch.

import contextlib
import datetime
import subprocess
import traceback

LOG_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
LOG_PATH = f'{LOG_BASE}/run_log.txt'
_failures = []  # [(step_name, exception_repr), ...] -- checked by Cell 6

def log(msg):
    line = f'[{datetime.datetime.now().isoformat(timespec="seconds")}] {msg}'
    print(line)
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
        f.flush()
        os.fsync(f.fileno())

@contextlib.contextmanager
def step(name):
    log(f'START   {name}')
    try:
        yield
    except Exception as e:
        log(f'FAILED  {name}\n{traceback.format_exc()}')
        _failures.append((name, repr(e)))
        print(f'>>> {name} FAILED -- continuing so later cells (esp. Cell 6, Save Results) '
              f'still run and record this. See {LOG_PATH} for the full traceback. <<<')
    else:
        log(f'DONE    {name}')

def require(*names):
    missing = [n for n in names if n not in globals() or globals()[n] is None]
    if missing:
        raise RuntimeError(
            f'Missing prerequisite(s): {missing} -- an earlier step must have '
            f'failed before setting these. Check run_log.txt for the FAILED '
            f'entry that comes before this one.'
        )

def run_shell(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    output = (result.stdout or '') + (result.stderr or '')
    log(f'$ {cmd}  (exit {result.returncode})\n{output}')
    print(output[-2000:])  # tail only in the cell's own live output; full text is in run_log.txt
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd} -- see {LOG_PATH} for full output')
    return result

log(f'=== New run -- log path: {LOG_PATH} ===')
print(f'Logging to {LOG_PATH}')


In [ ]:
# === Cell 3/6: Clone + install EasyEdit ===
# Not pip-installable in a way that ships its hparams/*.yaml files --
# ROMEHyperParams.from_hparams() reads those directly from a cloned repo,
# so cloning is required regardless of whether a PyPI easyeditor package
# also exists.
#
# Confirmed on a real run: installing EasyEdit's requirements.txt in one
# `pip install -r ...` shot failed -- a wheel-build error for some package
# partway through the ~30-package list aborted the whole install with no
# clear indication of which one.
#
# Confirmed on a real run, a second failure past the install itself: even
# after installing "successfully", `import easyeditor` later crashed with
# `ImportError: cannot import name '_center' from 'numpy._core.umath'`.
# Root cause: Kaggle's base image already has numpy/scipy/scikit-learn/
# torch/transformers preinstalled (torch already matched to the correct
# CUDA build) -- EasyEdit's ==-exact-pinned requirements.txt force-
# reinstalls all of these on top of an already-running Python process. A
# compiled extension already loaded against the OLD numpy's C ABI, combined
# with a freshly-installed DIFFERENT numpy version, breaks in exactly this
# way.
#
# Fixed: skip installing anything already importable in the current
# environment (checked via importlib.util.find_spec) rather than blindly
# reinstalling every pinned version -- avoids the numpy corruption, and is
# faster since foundational packages Kaggle already ships never get
# re-downloaded. Genuinely-missing packages still install normally, one at
# a time, logging (not raising) on a per-package failure so one bad package
# can't block the rest -- Cell 4's `import easyeditor` is the real test of
# whether a skipped-or-failed package actually mattered.

with step('clone + install EasyEdit'):
    EASYEDIT_DIR = os.path.join(os.path.dirname(REPO_DIR), 'EasyEdit')

    if os.path.exists(os.path.join(EASYEDIT_DIR, '.git')):
        run_shell(f'git -C {EASYEDIT_DIR} pull')
    else:
        run_shell(f'git clone https://github.com/zjunlp/EasyEdit.git {EASYEDIT_DIR}')

    req_path = os.path.join(EASYEDIT_DIR, 'requirements.txt')
    with open(req_path) as f:
        requirements = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]

    import importlib.util

    # pip package name -> the name you actually `import` -- only needed for
    # the ones that differ; everything else falls back to name.replace('-','_').
    _IMPORT_NAME = {
        'pyyaml': 'yaml', 'scikit-learn': 'sklearn', 'opencv-python': 'cv2',
        'hydra-core': 'hydra', 'importlib-metadata': 'importlib_metadata',
        'sentence-transformers': 'sentence_transformers',
    }

    failed_packages = []
    skipped_packages = []
    for req in requirements:
        pkg_name = req.split('==')[0].split('>=')[0].split('<')[0].strip()
        import_name = _IMPORT_NAME.get(pkg_name.lower(), pkg_name.replace('-', '_'))
        if importlib.util.find_spec(import_name) is not None:
            skipped_packages.append(req)
            log(f'  skipping {req} -- {import_name} already importable (Kaggle base image)')
            continue
        try:
            run_shell(f'pip install -q "{req}"')
        except RuntimeError:
            failed_packages.append(req)
            log(f'  (continuing past {req} -- Cell 4 will show whether this actually matters)')

    run_shell('pip install -q google-genai')

    if skipped_packages:
        log(f'Skipped {len(skipped_packages)} already-available package(s): {skipped_packages}')
    if failed_packages:
        log(f'WARNING: {len(failed_packages)} EasyEdit requirement(s) failed to install: {failed_packages}')
        print(f'>>> {len(failed_packages)} package(s) failed to install: {failed_packages}')
        print('    This may or may not matter -- Cell 4 (import easyeditor) will tell you for sure.')
    else:
        log('All remaining EasyEdit requirements installed successfully.')

    log(f'EasyEdit cloned to {EASYEDIT_DIR}')


In [ ]:
# === Cell 4/6: Load GPT-2 XL + ROME/MEMIT, then a cheap sanity check ===
# Loaded once, not per-conversation: WeightEditBaseline resets this same
# editor's weights back to pristine at the start of each conversation (see
# its __init__ and module docstring) rather than reloading the ~6GB model
# from HuggingFace every time.
#
# METHOD defaults to 'ROME': confirmed directly from both hparams files
# that ROME's gpt2-xl config sets mom2_adjustment: false (no covariance-
# statistics precompute needed) while MEMIT's sets it true with
# mom2_n_samples: 100000 -- a real, unmeasured extra cost. Get ROME's
# number first; MEMIT is this same interface, not a separate notebook.
METHOD = 'ROME'  # 'ROME' | 'MEMIT'

# Set before loading the model, per PyTorch's own suggestion in a real CUDA
# OOM this notebook hit below -- cheap and harmless to set proactively even
# if it turns out not to be the fix.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

with step(f'load GPT-2 XL + {METHOD}'):
    from baselines.easyedit_weight_editor import EasyEditWeightEditor

    editor = EasyEditWeightEditor(easyedit_dir=EASYEDIT_DIR, method=METHOD)
    log('Loading GPT-2 XL (first run downloads ~6GB from HuggingFace)...')
    editor.load()
    log('Loaded.')

# Cheap sanity check -- confirm editing actually works before spending real
# time on the benchmark. Confirmed on a real run: this specific step hit
# torch.OutOfMemoryError on a T4 (GPT-2 XL loaded fine at ~6GB, but ROME's
# compute_v optimization step -- forward+backward through all 48 layers, 20
# gradient-descent steps, for a single edit -- pushed usage to essentially
# the full 14.5GB available). PYTORCH_ALLOC_CONF above addresses memory
# *fragmentation*, but the error showed only ~317MB reserved-but-unallocated
# -- not enough slack for fragmentation alone to explain a 14MB shortfall,
# so this may not actually be the fix, just the cheap thing to try first.
# If this still OOMs, that's a real, open finding worth reporting as-is:
# full fp32 ROME editing on GPT-2 XL may genuinely not fit a single T4's
# 16GB with EasyEdit's default settings.
#
# Deliberately not a LoCoMo question -- a well-known fact whose default
# completion is predictable, so a wrong BEFORE or unchanged AFTER is
# obviously a real problem rather than model uncertainty.
with step('single-edit sanity check'):
    prompt = "The Eiffel Tower is located in the city of"

    before = editor.generate(prompt, max_tokens=8)
    log(f'BEFORE edit: {before!r}')

    editor.edit(prompt=prompt, subject="The Eiffel Tower", target_new="Rome")
    after_edit = editor.generate(prompt, max_tokens=8)
    log(f'AFTER edit:  {after_edit!r}')

    editor.reset()
    after_reset = editor.generate(prompt, max_tokens=8)
    log(f'AFTER reset: {after_reset!r}')

print()
print('>>> Check the three lines above before continuing:')
print('    AFTER edit should differ from BEFORE and reflect the new fact.')
print('    AFTER reset should return to (approximately) BEFORE.')
print('    If either does not hold, stop and debug -- do not proceed to Cell 6.')


In [ ]:
# === Cell 5/6: Gemini extraction client ===
# Used only for extraction and edit-request conversion (turning a free-text
# memory into a ROME-style prompt/subject/target_new) -- never for QA
# answers, which come from the edited model itself. Keeps everything except
# the model actually being edited API-based, no second heavy local model to
# load alongside GPT-2 XL.
#
# Add your key to Kaggle Secrets as GEMINI_API_KEY and enable it for this
# notebook, or paste it directly below for a quick run (remove it again
# afterward if you do).
with step('load Gemini extraction client'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
        log('Loaded GEMINI_API_KEY from Kaggle Secrets')
    except Exception:
        if not os.environ.get('GEMINI_API_KEY'):
            log('No GEMINI_API_KEY in Kaggle Secrets or the environment -- set one before continuing:')
            print("  os.environ['GEMINI_API_KEY'] = '...'")

    from llm.gemini_client import GeminiTextLLM
    extraction_llm = GeminiTextLLM()
    log(f'Extraction LLM: {extraction_llm.model_name}')


In [ ]:
# === Cell 6/6: Run WeightEditBaseline through the real LoCoMo benchmark, then save ===
# Same run_locomo harness, same F1 scoring, as every other system's LoCoMo
# number in this project. pipeline_factory closes over the single shared
# editor loaded in Cell 4 -- WeightEditBaseline.__init__ resets it to
# pristine weights for every new conversation.
#
# Scope starts deliberately small: every extracted, editable memory is a
# real ROME/MEMIT edit -- gradient descent against the actual model, not a
# cheap dict write -- so per-turn cost is categorically different from a
# retrieval baseline's QA step. NUM_CONVERSATIONS=1 and
# MAX_QA_PER_CONVERSATION=15 first, to get a real wall-clock/turn reading
# before committing to anything close to a full 10-conversation run.
#
# MAX_TURNS_PER_CONVERSATION=60 is a real, confirmed-necessary cap, not a
# cautious guess: extraction_llm is API-backed, so there's no way to
# combine turns into fewer requests the way a local model's batched forward
# pass would. A real run ingesting all ~400+ turns of one conversation hit
# the free tier's confirmed 15-requests/minute cap partway through, then
# its 429-retry fallback's own confirmed daily cap shortly after -- neither
# recoverable by waiting. 60 turns at the throttled ~4.5s/request pace is
# ~4.5 minutes of extraction calls, comfortably inside the per-minute limit
# with headroom left in the daily one too.
from benchmarks.locomo.evaluate import run_locomo
from baselines.weight_edit_baseline import WeightEditBaseline
import time

NUM_CONVERSATIONS = 1
MAX_QA_PER_CONVERSATION = 15
MAX_TURNS_PER_CONVERSATION = 60

def _weight_edit_factory(llm):
    return WeightEditBaseline(llm_client=llm, editor=editor)

CHECKPOINT_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
CHECKPOINT_PATH = f'{CHECKPOINT_BASE}/locomo_checkpoint_weightedit_{METHOD.lower()}.json'

locomo_result = None
_elapsed = None

with step(f'run_locomo (WeightEdit-{METHOD}, {NUM_CONVERSATIONS} conv, '
          f'{MAX_QA_PER_CONVERSATION} qa/conv, {MAX_TURNS_PER_CONVERSATION} turns/conv)'):
    require('editor', 'extraction_llm')
    log(f'Checkpoint path: {CHECKPOINT_PATH}')
    _t0 = time.time()
    locomo_result = run_locomo(
        llm_client=extraction_llm,
        num_conversations=NUM_CONVERSATIONS,
        max_qa_per_conversation=MAX_QA_PER_CONVERSATION,
        max_turns_per_conversation=MAX_TURNS_PER_CONVERSATION,
        checkpoint_path=CHECKPOINT_PATH,
        pipeline_factory=_weight_edit_factory,
        system_name=f'WeightEdit-{METHOD}',
    )
    _elapsed = time.time() - _t0
    log(f"avg F1: {locomo_result['avg_f1']:.1%}  (over {locomo_result['total']} questions)  "
        f"bins: {locomo_result['bins']}  wall_clock: {_elapsed:.0f}s")

# Written defensively: if Cell 6's own run_locomo step failed, locomo_result
# stays None instead of raising a second, compounding NameError/TypeError
# on top of the one step() already caught and logged.
WEIGHTEDIT_RESULTS_PATH = (
    ('/kaggle/working' if os.path.exists('/kaggle/working') else '/content')
    + f'/hippovoice_results_weightedit_{METHOD.lower()}.json'
)

import json
import datetime

weightedit_out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'system': f'WeightEdit-{METHOD}',
    'extraction_llm': extraction_llm.model_name if 'extraction_llm' in dir() else 'not reached',
    'num_conversations': globals().get('NUM_CONVERSATIONS', 'not reached'),
    'max_qa_per_conversation': globals().get('MAX_QA_PER_CONVERSATION', 'not reached'),
    'max_turns_per_conversation': globals().get('MAX_TURNS_PER_CONVERSATION', 'not reached'),
    'wall_clock_seconds': _elapsed if locomo_result is not None else 'not reached',
    'locomo': (
        {
            'avg_f1': locomo_result['avg_f1'],
            'total': locomo_result['total'],
            'bins': locomo_result['bins'],
            'details': locomo_result['details'],
        }
        if locomo_result is not None
        else 'not reached -- see run_log.txt for where the run stopped'
    ),
    'failures': [f'{name}: {exc}' for name, exc in _failures],
}

with open(WEIGHTEDIT_RESULTS_PATH, 'w') as f:
    json.dump(weightedit_out, f, indent=2)
log(f'Saved to {WEIGHTEDIT_RESULTS_PATH}')

# Marker lines specifically so this is grep-able out of whatever else
# Kaggle's own cell output/log noise surrounds it when pasted back.
print('===WEIGHTEDIT_RESULTS_JSON_START===')
print(json.dumps(weightedit_out, indent=2))
print('===WEIGHTEDIT_RESULTS_JSON_END===')

if _failures:
    print(f'\n{len(_failures)} step(s) failed -- see {LOG_PATH} for full tracebacks:')
    for name, exc in _failures:
        print(f'  - {name}: {exc}')
